This notebook will be used to figure out how I want to format the HDF5 file for the interpolated SNIa-SALT2 generated data. Most of this notebook is similar to CheckingICanUsePresto2Data.ipynb

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt

import Functions

from astropy.io import fits
from astropy.table import Table
from collections import OrderedDict as odict
import time
import math
from scipy import interpolate
import h5py

In [5]:
Path = '/lustre/lrspec/prestocolor/GSN_IDEAL_z02'

In [6]:
data = Functions.ReadData(Path, 'SNIa-SALT2', 0, 0)

In [7]:
data

MJD,BAND,CCDNUM,FIELD,PHOTFLAG,PHOTPROB,FLUXCAL,FLUXCALERR,PSF_SIG1,PSF_SIG2,PSF_RATIO,SKY_SIG,SKY_SIG_T,RDNOISE,ZEROPT,ZEROPT_ERR,GAIN,SIM_MAGOBS,SIM_FLUXCAL_HOSTERR
float64,bytes2,int16,bytes12,int32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32
53000.0,u,-9,NULL,0,-9.0,-0.0061405096,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53000.0,g,-9,NULL,0,-9.0,0.0036151197,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53000.0,r,-9,NULL,0,-9.0,-0.0020213476,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53000.0,i,-9,NULL,0,-9.0,0.0032451483,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53000.0,z,-9,NULL,0,-9.0,0.0051861512,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53000.0,Y,-9,NULL,0,-9.0,0.0036822045,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53002.0,u,-9,NULL,0,-9.0,-0.0036757744,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53002.0,g,-9,NULL,0,-9.0,0.0011087777,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0
53002.0,r,-9,NULL,0,-9.0,-0.004769873,nan,1.0,0.0,0.0,1264.911,0.0,31.622776,42.5,0.0,1.0,99.0,0.0


In [8]:
PathsDict = Functions.GetEventPaths(Path)
EventNames = list(PathsDict.keys())

In [9]:
#parameter setting
EventNames = EventNames

Bands = ['u ', 'g ', 'r ', 'i ', 'z ', 'Y ']

StartFile = 0
EndFile = None

StartObject = 0
EndObject = None

In [10]:
# for EventName in EventNames:
# only doing it for SNe Ia, so commenting out the above and replacing every instance of EventName with "SNIa-SALT2"

print('Processing {:<25}'.format("SNIa-SALT2"+':'), end='')
start = time.time()

SNIaFits = {'u': [], 'g': [], 'r': [], 'i': [], 'z': [], 'Y': []}

EventPath = PathsDict["SNIa-SALT2"]
FileNames = os.listdir(EventPath)
FileNames.sort()

for FileName in FileNames[StartFile: EndFile]:

    Ind = FileName.find('HEAD')

    if Ind > -1:

        print('|', end='')

        FileNamePHOT = FileName[:Ind] + 'PHOT.FITS.gz'

        HeadFilePath = os.path.join(EventPath, FileName)
        PhotFilePath = os.path.join(EventPath, FileNamePHOT)
        Data = Functions.read_snana_fits(HeadFilePath, PhotFilePath)

        # note that there is a header file and a photometry file from the Snana simulations, so that's something I need to take into account when trying
        # generalize my code

        for II, Obj in enumerate(Data[StartObject: EndObject]):

            for Band in Bands:

                Mask = Obj['BAND']==Band

                #THIS IS WHERE INTERPOLATION HAPPENS
                fobject  = interpolate.interp1d(Obj['MJD'][Mask], Obj['SIM_MAGOBS'][Mask])
                #interp1d legacy function, replace with different 1d interpolator
                #don't know which one, good question to ask "I noticed that scipy.interpolate.interp1d is a legacy function now,
                #is there a particular interpolation function I should use?"
                SNIaFits[Band[0]].append(fobject)

TimeRange = [Obj['MJD'][Mask][0], Obj['MJD'][Mask][-1]]

# with open(EventName+'_Fit.pkl', 'wb') as f:
#     pickle.dump(SNIaFits, f)
#     pickle.dump(TimeRange, f )
# something I need to look into

end = time.time()
print('\t time spent: {0:6.3f} s'.format(end-start))


Processing SNIa-SALT2:              ||||||||||||||||||||	 time spent: 61.778 s


In [21]:
len(SNIaFits["u"])

40000

In [14]:
#code to save dict as hdf5 group from YScharf at https://stackoverflow.com/questions/16494669/how-to-store-dictionary-in-hdf5-dataset#
hf = h5py.File('SNIa-SALT2_interp.h5', 'w') #keep file name same (except for extension) as for pickle files


In [23]:
# #dict_group = hf.create_group('SNIa-SALT2 Light Curves')
# for k, v in SNIaFits.items():
#     dict_group[k] = v
# hf.close()
# can't save interp objects as file -> should just focus on data cube next then